# Practical 3: Training an AI Designer

On Monday you built a surrogate: something that predicts lift and drag
in microseconds instead of seconds. Today you point an optimiser at it
and ask for the best wing section it can find.

Then we check the answer against a better model of reality, and see
whether your surrogate was telling the truth.

**You will edit two things all afternoon**: the *bounds* you give the
agent, and the *objective* it maximises. Everything else is provided.

---
### Before you start
Run the setup cell, then the contract check. If the contract fails you
will be handed the reference surrogate and you can do the whole
practical with it. Say which one you are using when you report.

In [ ]:
# Clone repo
%cd /content
! [ -d "/content/practical" ] && echo "Repository already cloned" || git clone https://github.com/ArnauMiro/BIP-Torino-practical.git practical
%cd /content/practical

# Colab setup. Skip locally if you already have these.
%pip install -r requirements.txt

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import P1_ancillary as bip
import P3_ancillary as p3
import P3_physics as phys

np.set_printoptions(precision=3, suppress=True)
print('clamp:', bip.CLAMP)
print('design angle:', p3.ALPHA_DESIGN, 'deg')

GROUP     = 'your-group-name'  # Your group name, for submission
CAMPAIGN  = 'your-data'        # Your data name

In [ ]:
# Only if you are loading a zipped module
import os
os.system(f"unzip {GROUP}.zip")

In [ ]:
# Your surrogate from Monday, or the reference if it will not load.
surrogate, which = p3.load_surrogate(GROUP) # Path to the model you generated
print(f'using the {which} surrogate')

assert p3.check_p3_contract(surrogate)

# Wrapped so we can count how many predictions each optimiser spends.
surrogate = p3.CallCounter(surrogate)

---
## Part 1. What does your surrogate think the best wing is?

Before any reinforcement learning, use a classical optimiser.
Differential evolution is a good baseline: no training, no
hyperparameters worth arguing about, and it will find the maximum of a
three-parameter function without much trouble.

**Predict before you run it.** Write down the section you expect:
camber, position, thickness. You will be asked about it later.

In [ ]:
objective = p3.LiftToDragObjective(surrogate)   # no fixes yet

surrogate.reset_counts()
t0 = time.time()
de = p3.run_de(objective, pop_size=20, n_gen=40, seed=0)

print(f'{de["shape"].name()}   m={de["params"][0]:.2f}  '
      f'p={de["params"][1]:.2f}  t={de["params"][2]:.2f}')
print(f'your surrogate promises L/D = {de["objective"]:.1f}')
print(f'{de["n_eval"]} evaluations in {time.time() - t0:.1f} s')

### Now ask something better

The referee is a higher-fidelity model than the one that generated your
training data. It is slow, so you would never optimise against it, but
you can afford to *check* against it.

Run the next cell and look at the gap.

In [ ]:
referee = phys.Referee()

gap = phys.promise_gap(surrogate.surrogate,
                       de['shape'].query(p3.ALPHA_DESIGN), referee)
print(f'promised   {gap["promised"][0]:8.1f}')
print(f'delivered  {gap["delivered"][0]:8.1f}')
print(f'PROMISE GAP{gap["gap"][0]:8.2f}')
print(f'inside your envelope: {gap["in_envelope"][0]}')

**Stop and think about that last line before continuing.**

Your envelope, the thing you built on Monday to tell you when the
surrogate is out of its depth, says this design is fine. Discuss in
your group what that means, and what an envelope can and cannot detect.

---
## Part 2. Train an agent

Same problem, different tool. Instead of searching for one good section,
train a *policy*: something that takes a section and improves it.

The environment gives the agent a random starting airfoil and lets it
nudge (m, p, t) for 64 steps. Reward is the improvement in the
objective, the same objective DE just used, the same Python object.

Start with the smoke test. If it fails, do not run the long cell.

In [ ]:
# SMOKE TEST — should finish in well under a minute.
# If the import line fails, note that pyLOM exports the *module*
# `shape_optimization_env`, not the class; P3_ancillary handles that
# for you, so use p3.make_env rather than importing it yourself.
from stable_baselines3 import PPO

p3.set_seeds(0)
env = p3.make_env(objective, episode_max_length=64)
obs, info = env.reset(seed=0)
print('observation:', obs, obs.dtype)
print('action space:', env.action_space)

t0 = time.time()
smoke = PPO('MlpPolicy', env, verbose=0, seed=0, **p3.PPO_KWARGS)
smoke.learn(total_timesteps=2000)
dt = time.time() - t0
print(f'2000 steps in {dt:.1f} s')
print(f'-> {p3.TRAIN_STEPS} steps ~ {dt * p3.TRAIN_STEPS / 2000 / 60:.1f} min')

In [ ]:
# The real run. verbose=1 so you can watch ep_rew_mean -- the endpoint
# is a bad guide here, because even a barely-trained policy reaches the
# clamp corner. Judge the curve, not the final section.
p3.set_seeds(0)
venv = p3.make_vec_env(objective, n_envs=8, episode_max_length=64, seed=0)

surrogate.reset_counts()
t0 = time.time()
model = PPO('MlpPolicy', venv, verbose=1, seed=0, **p3.PPO_KWARGS)
model.learn(total_timesteps=p3.TRAIN_STEPS)
train_s = time.time() - t0
print(f'trained in {train_s:.0f} s using '
      f'{surrogate.n_rows} surrogate evaluations')
model.save('{GROUP}_agent')

In [ ]:
# Fly the trained policy from a fixed start so the score is comparable.
eval_env = p3.make_env(objective, episode_max_length=64)
traj = p3.rollout(model, eval_env,
                  initial_shape=p3.NACA4Shape([0.0, 0.4, 12.0]))

best = traj['best']
print(f'agent -> {p3.NACA4Shape(best).name()}  {best}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(traj['objective'])
ax.set_xlabel('step'); ax.set_ylabel('objective'); ax.grid(alpha=.3)
ax.set_title('one episode, starting from NACA 0012')
plt.show()

In [ ]:
print(p3.comparison_table(
    {'DE': de['params'], 'RL agent': best},
    referee=referee, surrogate=surrogate.surrogate))

### The question that matters

DE almost certainly matched or beat the agent, using a tiny fraction of
the evaluations. Write down why you would ever use RL here.

If your answer is "you would not", that is a defensible answer for
*this* problem, hold onto it until the last section.

Optional: make a GIF of the agent morphing the section.

In [ ]:
p3.animate_evolution(traj['shapes'], traj['objective'],
                     path='agent.gif', title='the agent designing')

---
## Part 3. The fix ladder

Four things you can do about the promise gap. Try them in order and
record what each one buys you. **One of them will do nothing at all**,
and working out why is the most useful thing you will do today.

| rung | what it does | what you change |
|---|---|---|
| 1 | keep the agent inside the box your data covers | parameterizer bounds |
| 2 | impose a design requirement | `min_t`, `cl_min` |
| 3 | penalise leaving the envelope | `w_envelope` |
| 4 | spend a few referee calls and correct | `mf_loop` |

Rungs 1–3 use only what you already had on Monday. Rung 4 is the only
one that consults anything new.

In [ ]:
# Rung 1: the box your campaign actually covered.
X_camp, *_ = bip.load_campaign(CAMPAIGN) # Path to the data you generated
tight = {k: (float(X_camp[:, i].min()), float(X_camp[:, i].max()))
         for i, k in enumerate(('m', 'p', 't'))}
tight['alpha'] = bip.CLAMP['alpha']
print('your campaign covers:', {k: np.round(v, 2) for k, v in tight.items()})
print('the full clamp is:   ', bip.CLAMP)

r1 = p3.run_de(objective, bounds=tight, seed=0)
print(p3.comparison_table({'rung 0': de['params'], 'rung 1': r1['params']},
                          referee=referee, surrogate=surrogate.surrogate))

In [ ]:
# Rung 2: a structural requirement. Nothing to do with machine learning.
obj2 = p3.LiftToDragObjective(surrogate, min_t=10.0)
r2 = p3.run_de(obj2, seed=0)
print(p3.comparison_table({'rung 0': de['params'], 'rung 2': r2['params']},
                          referee=referee, surrogate=surrogate.surrogate))

In [ ]:
# Rung 3: penalise designs your envelope distrusts.
obj3 = p3.LiftToDragObjective(surrogate, w_envelope=200.0)
r3 = p3.run_de(obj3, seed=0)
print(p3.comparison_table({'rung 0': de['params'], 'rung 3': r3['params']},
                          referee=referee, surrogate=surrogate.surrogate))

terms = obj3.terms(np.array([de['params']]))
print('\nenvelope penalty on the rung-0 design:',
      terms['penalty_envelope'][0])

**Why did rung 3 change nothing?**

Do not move on until your group has an answer. Look at the penalty
printed above, and at `in_envelope` from Part 1.

Hint, if you are stuck: an envelope is built from *where your data is*.
What is it assuming about the data itself?

In [ ]:
# Rung 4: stop guessing, ask the referee.
res4 = p3.mf_loop(surrogate.surrogate, referee=referee,
                  rounds=3, n_per_round=25, seed=0)

for h in res4['history']:
    print(f"round {h['round']}  t={h['params'][2]:5.2f}  "
          f"promised {h['promised']:6.1f}  delivered {h['delivered']:6.1f}  "
          f"gap {h['gap']:.2f}   ({h['referee_calls']} referee calls)")

In [ ]:
print(p3.comparison_table(
    {'rung 0 naive': de['params'], 'rung 1 bounds': r1['params'],
     'rung 2 requirement': r2['params'], 'rung 3 envelope': r3['params'],
     'rung 4 referee': res4['params']},
    referee=referee, surrogate=surrogate.surrogate))

---
## Part 4. Submit

One design. It will be refereed live and put on the wall, ranked on what
it **delivers**, not on what your surrogate promised.

Also write one sentence: which rung earned you the most, and why.

In [ ]:
SUBMISSION = res4['params']        # or whichever you trust most

print(GROUP, p3.NACA4Shape(SUBMISSION).name(), np.round(SUBMISSION, 3))

---
## Stretch. One policy, many flight conditions

Everything so far had a fixed angle of attack, and DE beat the agent at
it. Here is the problem where that changes.

The angle is now drawn fresh each episode and appended to the
observation. The agent cannot change it, it has to design *for* it. So
you are no longer asking for one wing, you are asking for a rule that
produces the right wing for whatever condition it is handed.

DE can still win at any single angle. But it has to be re-run for every
one, while the policy answers in a single forward pass. **That** is the
comparison worth making, and it is the honest argument for RL on design
problems.

In [ ]:
p3.set_seeds(0)
cenv = p3.make_contextual_env(objective, episode_max_length=64)
print('observation is now', cenv.observation_space.shape[0],
      'numbers: m, p, t, alpha')

# Harder problem, four observations instead of three, so give it
# more budget than the single-condition run.
cmodel = PPO('MlpPolicy', cenv, verbose=1, seed=0, **p3.PPO_KWARGS)
cmodel.learn(total_timesteps=2 * p3.TRAIN_STEPS)

In [ ]:
# One policy, three conditions. Compare against DE re-run per angle.
# NOTE the reset_options: without it the angle is redrawn at random
# and you are not measuring the condition you think you are.
rows, cost = {}, {'policy': 0, 'DE': 0}
for a in (2.0, 5.0, 8.0):
    t = p3.rollout(cmodel, cenv,
                   initial_shape=p3.NACA4Shape([0.0, 0.4, 12.0]),
                   reset_options={'alpha': a})
    rows[f'policy @ {a:.0f} deg'] = t['best'][:3]
    d = p3.run_de(p3.LiftToDragObjective(surrogate, alpha=a), seed=0)
    rows[f'DE      @ {a:.0f} deg'] = d['params']
    cost['DE'] += d['n_eval']

for k, v in rows.items():
    print(f'{k:<20} {p3.NACA4Shape(v).name()}  {np.round(v, 2)}')
print(f"\nDE spent {cost['DE']} evaluations for three angles.")
print('The policy spent one forward pass each, after training once.')

Last question, and it is the one to leave with:

How many evaluations did the policy need? How many did DE need for three
angles? For thirty angles? At what point does training a policy stop
being an indulgence?